<a href="https://colab.research.google.com/github/paradoxailab/paradox-world-model/blob/atlas-v0.1/atlas/notebooks/atlas_threshold_grid_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atlas threshold grid on WEATHER-5K (ParadoxAI Lab)

Rules written before these runs: `atlas/README.md`, sections **WB1-T** (2026-11-16) and **Head-to-head on WEATHER-5K and
the minimum threshold function**, Phases 2–3 (2026-11-17), with the clarifications in **Threshold grid on Colab**
(2026-11-22). Validation 2022 only; the test year is not touched.

**Run:** Runtime → GPU (A100 or L4) → Run all, allow Google Drive. Everything resumes after a disconnect: finished runs
are skipped, an interrupted run continues from its last checkpoint. About 19 A100-hours in all; the WB1-T runs (the
answer for 1M) come first, about 2 hours.

In [1]:
# 1. Drive, data, code
import os, shutil, glob, json, subprocess
from google.colab import drive
drive.mount('/content/drive', timeout_ms=300000)
W = '/content/drive/MyDrive/PWM-10M'; OUT = f'{W}/thr'; os.makedirs(OUT, exist_ok=True)
R = '/content/raw'; os.makedirs(R, exist_ok=True)
assert os.path.exists(f'{W}/data/W5K_X.npy'), 'the WEATHER-5K copy (PWM-10M/data) is missing from Drive'
if not (os.path.exists(f'{R}/W5K_X.npy') and os.path.getsize(f'{R}/W5K_X.npy') == os.path.getsize(f'{W}/data/W5K_X.npy')):
    shutil.copy(f'{W}/data/W5K_META.npz', R); shutil.copy(f'{W}/data/W5K_X.npy', R)
!rm -rf /content/pwm && git clone -q --depth 1 -b atlas-v0.1 https://github.com/paradoxailab/paradox-world-model.git /content/pwm
!cd /content/pwm && git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv
# the four WB1-T runs already done on Kaggle (1/64, 1/16, 1/4, all) are points of the grid's 1M row
for f in glob.glob('/content/pwm/atlas/results/kaggle/atlas-w5k-thr-a/w5k_atlas_1m_h24_f*_s600.json') + \
         ['/content/pwm/atlas/results/kaggle/atlas-w5k-atlas1m-h24/w5k_atlas_1m_h24_s600.json']:
    if not os.path.exists(f'{OUT}/{os.path.basename(f)}'):
        shutil.copy(f, OUT)
print(sorted(os.listdir(OUT)))

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# 2. The runs, in order: WB1-T (1M at 1/32, 1/8, 1/2), then the grid by size, then the second seeds.
#    Same recipe as the WB1-T runs (run_atlas_5k.py defaults, seed 600), validation only.
os.environ['PW_RAW'] = R; os.environ['ATLAS_OUT'] = OUT
lab = {100_000: '100k', 300_000: '300k', 3_000_000: '3m', 10_000_000: '10m'}
def sfx(fr): return '' if fr == 1 else '_f' + str(fr).replace('.', '')
JOBS = [(f'w5k_atlas_1m_h24{sfx(fr)}', 1_000_000, fr, 600) for fr in (0.03125, 0.125, 0.5)]
for size in (100_000, 300_000, 3_000_000, 10_000_000):
    for fr in (0.015625, 0.0625, 0.25, 1):
        JOBS.append((f'w5k_grid_{lab[size]}_h24{sfx(fr)}', size, fr, 600))
JOBS += [('w5k_seed2_atlas_1m_h24', 1_000_000, 1, 601), ('w5k_seed2_grid_10m_h24', 10_000_000, 1, 601)]
for tag, size, fr, seed in JOBS:
    res = f'{OUT}/{tag}_s{seed}.json'
    while not os.path.exists(res):
        cmd = ['python', '-u', 'run_atlas_5k.py', '--model', 'atlas', '--H', '24', '--size', str(size), '--seed', str(seed),
               '--tag', tag, '--no-test', '--pause-at-cap', '--cap', '600', '--max-params', '11000000']
        if fr != 1:
            cmd += ['--frac', str(fr)]
        print('RUN', tag, flush=True)
        p = subprocess.run(cmd, cwd='/content/pwm/atlas/delhi', capture_output=True, text=True)
        open(f'{OUT}/log_{tag}.txt', 'a').write(p.stdout + p.stderr)
        print('\n'.join([l for l in p.stdout.splitlines() if 'val' in l.lower() or 'param' in l.lower()][-4:]), flush=True)
        if p.returncode != 0 and not os.path.exists(res):
            print(p.stderr[-2000:]); raise SystemExit(f'{tag} failed')
    print('DONE', tag, json.load(open(res)).get('val_best'), flush=True)

In [ ]:
# 3. Scores: WB1-T (the 1M data threshold function) and the grid (Phase 3: forms A/B/C, held-out cells, MTF)
%cd /content/pwm/atlas/delhi
!ATLAS_OUT={OUT} python -u score_w5k_thr.py 2>&1 | tee {OUT}/log_w5k_thr_score.txt
!ATLAS_OUT={OUT} python -u score_w5k_grid.py 2>&1 | tee {OUT}/log_w5k_grid_score.txt